#Creating the Dataset for the exercise
##Create_dataset.py to generate a realistic e-commerce dataset

In [1]:
# ============================================================
# ecommerce_sales.csv — Dataset Generator (fully fixed)
# ============================================================
import csv
import random
from datetime import datetime, timedelta

random.seed(42)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
NUM_RECORDS  = 2000
START_DATE   = datetime(2024, 1, 1)

# 🔑 NOTE: categories is a DICT (not a list!)
#    so that categories["Clothing"] works.
categories = {
    "Electronics": ["Laptop", "Phone", "Headphones", "Tablet", "Smartwatch"],
    "Clothing":    ["T-Shirt", "Jeans", "Jacket", "Shoes", "Hat"],
    "Home":        ["Blender", "Lamp", "Chair", "Pillow", "Vacuum"],
    "Sports":      ["Yoga Mat", "Dumbbell", "Bike", "Tennis Racket", "Ball"],
    "Books":       ["Novel", "Textbook", "Comic", "Biography", "Cookbook"],
    "Beauty":      ["Lipstick", "Perfume", "Cream", "Shampoo", "Serum"],
}

# 🔑 price_ranges MUST have the SAME keys as `categories`
price_ranges = {
    "Electronics": (50, 1500),
    "Clothing":    (10, 200),
    "Home":        (15, 400),
    "Sports":      (10, 300),
    "Books":       (5, 60),
    "Beauty":      (5, 120),
}

regions          = ["North", "South", "East", "West", "Central"]
payment_methods  = ["Credit Card", "PayPal", "Debit Card",
                    "Cash on Delivery", "Gift Card"]
channels         = ["Web", "Mobile App", "In-Store", "Phone"]

# ------------------------------------------------------------
# ROW GENERATOR FUNCTION
# ------------------------------------------------------------
def generate_record(order_id):
    # 1. Pick a category, then a product from that category
    category = random.choice(list(categories.keys()))     # e.g. "Clothing"
    product  = random.choice(categories[category])        # ✅ dict lookup works

    # 2. Price from the matching price range
    min_p, max_p = price_ranges[category]
    unit_price = round(random.uniform(min_p, max_p), 2)

    # 3. Quantity, discount, total
    quantity = random.randint(1, 5)
    discount = round(random.uniform(0.0, 0.4), 2)
    total    = round(unit_price * quantity * (1 - discount), 2)

    # 4. Order date (spread across 2024)
    order_date = START_DATE + timedelta(days=random.randint(0, 365))

    # 5. Satisfaction (1–5) — higher for discounted items
    base_satisfaction = random.choices(
        [1, 2, 3, 4, 5], weights=[5, 10, 20, 35, 30]
    )[0]
    if discount > 0.1:
        base_satisfaction = min(5, base_satisfaction + 1)

    # 6. Return the row as a dict
    return {
        "order_id":       f"ORD{order_id:06d}",
        "order_date":     order_date.strftime("%Y-%m-%d"),
        "customer_id":    f"CUST{random.randint(1, 2000):05d}",
        "category":       category,
        "product":        product,
        "unit_price":     unit_price,
        "quantity":       quantity,
        "discount":       discount,
        "total_amount":   total,
        "region":         random.choice(regions),
        "payment_method": random.choice(payment_methods),
        "channel":        random.choice(channels),
        "satisfaction":   base_satisfaction,
        "delivery_days":  random.randint(1, 14),
    }

# ------------------------------------------------------------
# GENERATE RECORDS
# ------------------------------------------------------------
records = [generate_record(i) for i in range(1, NUM_RECORDS + 1)]

# ------------------------------------------------------------
# WRITE TO CSV
# ------------------------------------------------------------
filename = "ecommerce_sales.csv"
with open(filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------
print(f"✅ Dataset created: {filename}")
print(f"   Total records:   {len(records)}")
print(f"   Columns:         {list(records[0].keys())}")
print("\n📄 Sample records:")
for r in records[:3]:
    print("  ", r)

✅ Dataset created: ecommerce_sales.csv
   Total records:   2000
   Columns:         ['order_id', 'order_date', 'customer_id', 'category', 'product', 'unit_price', 'quantity', 'discount', 'total_amount', 'region', 'payment_method', 'channel', 'satisfaction', 'delivery_days']

📄 Sample records:
   {'order_id': 'ORD000001', 'order_date': '2024-03-12', 'customer_id': 'CUST01386', 'category': 'Beauty', 'product': 'Lipstick', 'unit_price': 7.88, 'quantity': 3, 'discount': 0.1, 'total_amount': 21.28, 'region': 'Central', 'payment_method': 'Credit Card', 'channel': 'Phone', 'satisfaction': 5, 'delivery_days': 1}
   {'order_id': 'ORD000002', 'order_date': '2024-10-14', 'customer_id': 'CUST01331', 'category': 'Electronics', 'product': 'Laptop', 'unit_price': 367.03, 'quantity': 5, 'discount': 0.24, 'total_amount': 1394.71, 'region': 'Central', 'payment_method': 'Cash on Delivery', 'channel': 'Mobile App', 'satisfaction': 4, 'delivery_days': 8}
   {'order_id': 'ORD000003', 'order_date': '2024-03-

In [2]:
"""
BT80733 — Big Data Analytics
Day 4 Exercise: PySpark Analytics on E-Commerce Dataset
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    MulticlassClassificationEvaluator
)

# ============================================================
# INITIALISE SPARK
# ============================================================
spark = SparkSession.builder \
    .appName("Day4Exercise") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


C:\Users\user\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


# PART A: DATA LOADING AND EXPLORATION



Q1: Load csv with explicit schema

In [3]:
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("customer_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("product", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("discount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("region", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("satisfaction", IntegerType(), True),
    StructField("delivery_days", IntegerType(), True),
])

df = spark.read.csv("ecommerce_sales.csv", header=True, schema=schema)

# Q2: Schema and first 10 rows


In [4]:
print("=" * 60)
print("Q2: SCHEMA AND SAMPLE DATA")
print("=" * 60)
df.printSchema()
df.show(10, truncate=False)



Q2: SCHEMA AND SAMPLE DATA
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- region: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- satisfaction: integer (nullable = true)
 |-- delivery_days: integer (nullable = true)

+---------+----------+-----------+-----------+----------+----------+--------+--------+------------+-------+----------------+----------+------------+-------------+
|order_id |order_date|customer_id|category   |product   |unit_price|quantity|discount|total_amount|region |payment_method  |channel   |satisfaction|delivery_days|
+---------+----------+-----------+-----------+----------+-

# Q3: Total orders

In [5]:
print("=" * 60)
print("Q3: TOTAL ORDERS")
print("=" * 60)
print(f"Total orders: {df.count()}")

Q3: TOTAL ORDERS
Total orders: 2000


# Q4: Unique customers

In [6]:

print("=" * 60)
print("Q4: UNIQUE CUSTOMERS")
print("=" * 60)
print(f"Unique customers: {df.select('customer_id').distinct().count()}")

Q4: UNIQUE CUSTOMERS
Unique customers: 1246



# Q5: Date range

In [7]:
print("=" * 60)
print("Q5: DATE RANGE")
print("=" * 60)
df.agg(
    min("order_date").alias("Earliest"),
    max("order_date").alias("Latest")
).show()

Q5: DATE RANGE
+----------+----------+
|  Earliest|    Latest|
+----------+----------+
|2024-01-01|2024-12-31|
+----------+----------+



# PART B: DESCRIPTIVE ANALYTICS

# Q6: Total revenue


print("=" * 60)
print("Q6: TOTAL REVENUE")
print("=" * 60)
df.agg(round(sum("total_amount"), 2).alias("TotalRevenue")).show()


df.groupBy("product", "category") \
    .agg(round(sum("total_amount"), 2).alias("TotalRevenue")) \
    .orderBy(desc("TotalRevenue")) \
    .show(5)


# Q7: Average order value


In [8]:

print("=" * 60)
print("Q7: AVERAGE ORDER VALUE")
print("=" * 60)
df.agg(round(avg("total_amount"), 2).alias("AvgOrderValue")).show()



Q7: AVERAGE ORDER VALUE
+-------------+
|AvgOrderValue|
+-------------+
|       528.04|
+-------------+



# Q8: Top 5 products by revenue


In [9]:
print("=" * 60)
print("Q8: TOP 5 PRODUCTS BY REVENUE")
print("=" * 60)
df.groupBy("product", "category") \
    .agg(round(sum("total_amount"), 2).alias("TotalRevenue")) \
    .orderBy(desc("TotalRevenue")) \
    .show(5)


Q8: TOP 5 PRODUCTS BY REVENUE
+----------+-----------+------------+
|   product|   category|TotalRevenue|
+----------+-----------+------------+
|Headphones|Electronics|   153151.06|
|     Phone|Electronics|    120214.7|
|    Laptop|Electronics|   116539.35|
|    Tablet|Electronics|   114235.64|
|Smartwatch|Electronics|   107517.24|
+----------+-----------+------------+
only showing top 5 rows


# Q9: Revenue by category


In [10]:
print("=" * 60)
print("Q9: REVENUE BY CATEGORY")
print("=" * 60)
df.groupBy("category") \
    .agg(round(sum("total_amount"), 2).alias("TotalRevenue")) \
    .orderBy(desc("TotalRevenue")) \
    .show()

Q9: REVENUE BY CATEGORY
+-----------+------------+
|   category|TotalRevenue|
+-----------+------------+
|Electronics|   611657.99|
|       Home|   148713.08|
|     Sports|   127335.45|
|   Clothing|    86987.42|
|     Beauty|    54147.85|
|      Books|    27243.46|
+-----------+------------+



# Q10: Orders by region


In [11]:
print("=" * 60)
print("Q10: ORDERS BY REGION")
print("=" * 60)
df.groupBy("region") \
    .agg(
        count("*").alias("OrderCount"),
        round(sum("total_amount"), 2).alias("TotalRevenue")
    ) \
    .orderBy(desc("TotalRevenue")) \
    .show()

Q10: ORDERS BY REGION
+-------+----------+------------+
| region|OrderCount|TotalRevenue|
+-------+----------+------------+
|  North|       440|   252105.89|
|Central|       420|   244804.99|
|   East|       383|    196996.6|
|   West|       386|   183706.45|
|  South|       371|   178471.32|
+-------+----------+------------+



# PART C: CUSTOMER ANALYTICS


# Q11: Top 10 customers by spending

In [12]:
print("=" * 60)
print("Q11: TOP 10 CUSTOMERS BY SPENDING")
print("=" * 60)
df.groupBy("customer_id") \
.agg(round(sum("total_amount"), 2). alias("TotalSpent"))\
.orderBy(desc("TotalSpent"))\
.show(10)

Q11: TOP 10 CUSTOMERS BY SPENDING
+-----------+----------+
|customer_id|TotalSpent|
+-----------+----------+
|  CUST00529|   8855.42|
|  CUST01940|   7011.49|
|  CUST00030|   6690.51|
|  CUST01823|   6684.83|
|  CUST01936|   6471.96|
|  CUST01289|   6400.21|
|  CUST01785|   6361.89|
|  CUST01639|   6217.84|
|  CUST01327|   6095.07|
|  CUST00645|   6085.32|
+-----------+----------+
only showing top 10 rows


# Q12 Average satisfaction per category


In [13]:
print("=" * 60)
print("Q12: AVERAGE SATISFACTION PER CATEGORY")
print("=" * 60)
df.groupBy("category")\
.agg(round(avg("satisfaction"), 2).alias("AvgSatisfaction"))\
.orderBy(desc("AvgSatisfaction"))\
.show()


Q12: AVERAGE SATISFACTION PER CATEGORY
+-----------+---------------+
|   category|AvgSatisfaction|
+-----------+---------------+
|       Home|            4.3|
|     Sports|            4.3|
|      Books|           4.29|
|     Beauty|           4.27|
|Electronics|           4.25|
|   Clothing|           4.22|
+-----------+---------------+



# Q13: Payment methods

In [14]:
print("=" * 60)
print("Q13: PAYMENT METHOD ANALYSIS")
print("=" * 60)
df.groupBy("payment_method") \
    .agg(
        count("*").alias("OrderCount"),
        round(sum("total_amount"), 2).alias("TotalRevenue")
    ) \
    .orderBy(desc("OrderCount")) \
    .show()

Q13: PAYMENT METHOD ANALYSIS
+----------------+----------+------------+
|  payment_method|OrderCount|TotalRevenue|
+----------------+----------+------------+
|          PayPal|       430|   239667.67|
|       Gift Card|       402|   210834.64|
|Cash on Delivery|       394|   187123.77|
|      Debit Card|       391|   215370.39|
|     Credit Card|       383|   203088.78|
+----------------+----------+------------+




# Q14: Average delivery time per region

In [15]:
print("=" * 60)
print("Q14: AVERAGE DELIVERY TIME PER REGION")
print("=" * 60)
df.groupBy("region") \
    .agg(round(avg("delivery_days"), 2).alias("AvgDeliveryDays")) \
    .orderBy("AvgDeliveryDays") \
    .show()

Q14: AVERAGE DELIVERY TIME PER REGION
+-------+---------------+
| region|AvgDeliveryDays|
+-------+---------------+
|  North|           7.46|
|  South|           7.47|
|   East|           7.58|
|   West|            7.6|
|Central|           7.79|
+-------+---------------+



# Q15: Customers with > 5 orders


In [16]:
print("=" * 60)
print("Q15: CUSTOMERS WITH MORE THAN 5 ORDERS")
print("=" * 60)
frequent = df.groupBy("customer_id") \
    .agg(count("*").alias("OrderCount")) \
    .filter(col("OrderCount") > 5)
print(f"Number of frequent customers: {frequent.count()}")
frequent.orderBy(desc("OrderCount")).show(10)

Q15: CUSTOMERS WITH MORE THAN 5 ORDERS
Number of frequent customers: 2
+-----------+----------+
|customer_id|OrderCount|
+-----------+----------+
|  CUST01075|         6|
|  CUST01460|         6|
+-----------+----------+



# PART D: ADVANCED ANALYTICS WITH SPARK SQL

In [17]:

df.createOrReplaceTempView("sales")

# Q16: Monthly revenue trend

In [18]:
print("=" * 60)
print("Q16: MONTHLY REVENUE TREND")
print("=" * 60)
spark.sql("""
    SELECT
        DATE_FORMAT(order_date, 'yyyy-MM') AS Month,
        ROUND(SUM(total_amount), 2) AS TotalRevenue,
        COUNT(*) AS OrderCount
    FROM sales
    GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
    ORDER BY Month
""").show(12)


Q16: MONTHLY REVENUE TREND
+-------+------------+----------+
|  Month|TotalRevenue|OrderCount|
+-------+------------+----------+
|2024-01|    91532.13|       167|
|2024-02|   111414.09|       172|
|2024-03|     68130.0|       165|
|2024-04|    71868.64|       145|
|2024-05|    64043.01|       153|
|2024-06|    59242.12|       162|
|2024-07|    94386.34|       152|
|2024-08|    91814.62|       186|
|2024-09|    79562.61|       163|
|2024-10|   107423.72|       172|
|2024-11|   103265.51|       166|
|2024-12|   113402.46|       197|
+-------+------------+----------+



 # Q17: Top 3 products per category

In [19]:
print("=" * 60)
print("Q17: TOP 3 PRODUCTS PER CATEGORY")
print("=" * 60)
spark.sql("""
    WITH product_revenue AS (
        SELECT category, product, SUM(total_amount) AS revenue
        FROM sales
        GROUP BY category, product
    ),
    ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rn
        FROM product_revenue
    )
    SELECT category, product, ROUND(revenue, 2) AS revenue
    FROM ranked
    WHERE rn <= 3
    ORDER BY category, revenue DESC
""").show(30)


Q17: TOP 3 PRODUCTS PER CATEGORY
+-----------+----------+---------+
|   category|   product|  revenue|
+-----------+----------+---------+
|     Beauty|  Lipstick| 12986.34|
|     Beauty|   Shampoo|  11084.3|
|     Beauty|   Perfume| 10159.55|
|      Books|     Novel|  6554.31|
|      Books|  Textbook|  6126.94|
|      Books|  Cookbook|  5399.25|
|   Clothing|       Hat|  21616.0|
|   Clothing|    Jacket| 17528.53|
|   Clothing|   T-Shirt| 17350.09|
|Electronics|Headphones|153151.06|
|Electronics|     Phone| 120214.7|
|Electronics|    Laptop|116539.35|
|       Home|    Pillow| 33027.63|
|       Home|      Lamp| 30650.13|
|       Home|   Blender| 29858.98|
|     Sports|      Bike|  26559.0|
|     Sports|  Dumbbell| 26128.42|
|     Sports|      Ball| 25784.69|
+-----------+----------+---------+



# Q18: Correlation between discount and satisfaction

In [20]:
print("=" * 60)
print("Q18: CORRELATION — DISCOUNT vs SATISFACTION")
print("=" * 60)
corr_value = df.stat.corr("discount", "satisfaction")
print(f"Correlation(discount, satisfaction) = {corr_value:.4f}")


Q18: CORRELATION — DISCOUNT vs SATISFACTION
Correlation(discount, satisfaction) = 0.2653


# Q19: Busiest day of the week

In [21]:
print("=" * 60)
print("Q19: BUSIEST DAY OF THE WEEK")
print("=" * 60)
spark.sql("""
    SELECT
        DATE_FORMAT(order_date, 'EEEE') AS DayOfWeek,
        COUNT(*) AS OrderCount
    FROM sales
    GROUP BY DATE_FORMAT(order_date, 'EEEE')
    ORDER BY OrderCount DESC
""").show()


Q19: BUSIEST DAY OF THE WEEK
+---------+----------+
|DayOfWeek|OrderCount|
+---------+----------+
|   Monday|       309|
|   Sunday|       303|
|  Tuesday|       296|
| Saturday|       285|
|   Friday|       277|
|Wednesday|       271|
| Thursday|       259|
+---------+----------+



# Q20: Average order value by channel

In [22]:
print("=" * 60)
print("Q20: AVERAGE ORDER VALUE BY CHANNEL")
print("=" * 60)
spark.sql("""
    SELECT
        channel,
        ROUND(AVG(total_amount), 2) AS AvgOrderValue,
        COUNT(*) AS OrderCount
    FROM sales
    GROUP BY channel
    ORDER BY AvgOrderValue DESC
""").show()


Q20: AVERAGE ORDER VALUE BY CHANNEL
+----------+-------------+----------+
|   channel|AvgOrderValue|OrderCount|
+----------+-------------+----------+
|     Phone|       555.56|       505|
|  In-Store|       546.81|       484|
|       Web|       530.18|       499|
|Mobile App|       481.07|       512|
+----------+-------------+----------+



# PART E: MACHINE LEARNING WITH MLLIB


# --- Q21: Linear Regression — Predict total_amount ---


In [23]:
print("=" * 60)
print("Q21: LINEAR REGRESSION — PREDICT TOTAL AMOUNT")
print("=" * 60)

# Assemble features
assembler_lr = VectorAssembler(
    inputCols=["unit_price", "quantity", "discount"],
    outputCol="features"
)
lr_data = assembler_lr.transform(df).select("features", "total_amount")

# Split
train_lr, test_lr = lr_data.randomSplit([0.8, 0.2], seed=42)

# Train
lr = LinearRegression(featuresCol="features", labelCol="total_amount")
lr_model = lr.fit(train_lr)

# Predict
predictions_lr = lr_model.transform(test_lr)

# Evaluate
eval_rmse = RegressionEvaluator(
    labelCol="total_amount", predictionCol="prediction", metricName="rmse"
)
eval_r2 = RegressionEvaluator(
    labelCol="total_amount", predictionCol="prediction", metricName="r2"
)

rmse = eval_rmse.evaluate(predictions_lr)
r2 = eval_r2.evaluate(predictions_lr)

print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")
print(f"Coefficients: {lr_model.coefficients}")
print(f"Intercept: {lr_model.intercept:.4f}")

print("\nSample predictions:")
predictions_lr.select("features", "total_amount", "prediction").show(10)

Q21: LINEAR REGRESSION — PREDICT TOTAL AMOUNT
RMSE: 392.1056
R²: 0.7927
Coefficients: [2.379052117100579,168.71031601007172,-671.489344498741]
Intercept: -368.1076

Sample predictions:
+---------------+------------+-------------------+
|       features|total_amount|         prediction|
+---------------+------------+-------------------+
|[5.35,2.0,0.25]|        8.02|-185.83139825328374|
|[5.78,5.0,0.17]|       23.99|  375.0416897471839|
|[5.99,1.0,0.25]|        4.49| -353.0191209084111|
|[6.24,3.0,0.37]|       11.79| -95.58244719884146|
|[7.18,3.0,0.16]|       18.09|  47.66662413596873|
|[7.67,1.0,0.26]|        5.68| -355.7372067966695|
|[7.87,2.0,0.19]|       12.75| -139.5468262482658|
|[8.31,1.0,0.09]|        7.56|-240.06142487693918|
|[9.64,3.0,0.06]|       27.18| 120.66802679391026|
|[9.67,1.0,0.06]|        9.09|-216.68123366272016|
+---------------+------------+-------------------+
only showing top 10 rows


# --- Q22: Random Forest Classifier — Predict satisfaction ---

In [24]:
print("=" * 60)
print("Q22: RANDOM FOREST — PREDICT SATISFACTION")
print("=" * 60)

assembler_rf = VectorAssembler(
    inputCols=["total_amount", "delivery_days", "discount", "quantity"],
    outputCol="features"
)
rf_data = assembler_rf.transform(df).select("features", "satisfaction")

train_rf, test_rf = rf_data.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="satisfaction",
    numTrees=50,
    maxDepth=5,
    seed=42
)
rf_model = rf.fit(train_rf)
predictions_rf = rf_model.transform(test_rf)

eval_acc = MulticlassClassificationEvaluator(
    labelCol="satisfaction", predictionCol="prediction", metricName="accuracy"
)
accuracy = eval_acc.evaluate(predictions_rf)
print(f"Accuracy: {accuracy:.4f}")

print("\nConfusion Matrix:")
predictions_rf.groupBy("satisfaction", "prediction").count().orderBy("satisfaction", "prediction").show()

Q22: RANDOM FOREST — PREDICT SATISFACTION
Accuracy: 0.5475

Confusion Matrix:
+------------+----------+-----+
|satisfaction|prediction|count|
+------------+----------+-----+
|           1|       4.0|    3|
|           1|       5.0|    5|
|           2|       4.0|    5|
|           2|       5.0|   10|
|           3|       2.0|    1|
|           3|       4.0|    4|
|           3|       5.0|   44|
|           4|       2.0|    2|
|           4|       4.0|   19|
|           4|       5.0|   72|
|           5|       2.0|    2|
|           5|       3.0|    1|
|           5|       4.0|   13|
|           5|       5.0|  177|
+------------+----------+-----+



#----Q23: K -means clustering  - customer segmentation----

In [25]:
print("=" * 60)
print("Q23: K-MEANS CLUSTERING — CUSTOMER SEGMENTATION")
print("=" * 60)

# Build customer-level features
customer_features = df.groupBy("customer_id").agg(
    sum("total_amount").alias("TotalSpent"),
    count("*").alias("OrderFrequency"),
    avg("satisfaction").alias("AvgSatisfaction")
)

assembler_km = VectorAssembler(
    inputCols=["TotalSpent", "OrderFrequency", "AvgSatisfaction"],
    outputCol="features"
)
km_data = assembler_km.transform(customer_features)

# Try different k values
for k in [2, 3, 4, 5]:
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    model = kmeans.fit(km_data)
    wssse = model.summary.trainingCost
    print(f"k={k}, WSSSE={wssse:.2f}")

# Use k=4 (adjust based on results)
kmeans = KMeans(featuresCol="features", k=4, seed=42)
km_model = kmeans.fit(km_data)
clustered = km_model.transform(km_data)

print("\nCluster distribution:")
clustered.groupBy("prediction").count().orderBy("prediction").show()

print("\nCluster centroids:")
for i, center in enumerate(km_model.clusterCenters()):
    print(f"Cluster {i}: TotalSpent={center[0]:.2f}, Frequency={center[1]:.2f}, Satisfaction={center[2]:.2f}")


Q23: K-MEANS CLUSTERING — CUSTOMER SEGMENTATION
k=2, WSSSE=462381338.60
k=3, WSSSE=256973221.41
k=4, WSSSE=136964800.81
k=5, WSSSE=93382949.27

Cluster distribution:
+----------+-----+
|prediction|count|
+----------+-----+
|         0|  831|
|         1|   35|
|         2|  117|
|         3|  263|
+----------+-----+


Cluster centroids:
Cluster 0: TotalSpent=255.26, Frequency=1.36, Satisfaction=4.29
Cluster 1: TotalSpent=5336.02, Frequency=2.54, Satisfaction=4.11
Cluster 2: TotalSpent=2952.91, Frequency=2.05, Satisfaction=4.27
Cluster 3: TotalSpent=1185.23, Frequency=2.05, Satisfaction=4.22


# CLEANUP

In [26]:
spark.stop()
print("\n" + "=" * 60)
print("EXERCISE COMPLETE")
print("=" * 60)



EXERCISE COMPLETE
